# Notebook 2: Correlation Analysis

Analyze correlations between MSFT and other tech stocks.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_all_stocks
from src.data_cleaner import clean_data
from src.analyzer import calculate_correlation_matrix, identify_correlations
from src.visualizer import plot_correlation_heatmap
from config.config import PRIMARY_STOCK, ALL_STOCKS

print('✅ All imports successful!')


## Load and Prepare Data

In [ ]:
# Load stocks
stocks_data = load_all_stocks()

# Clean data
clean_stocks = {}
for ticker, data in stocks_data.items():
    clean_stocks[ticker] = clean_data(data, ticker)

print('✅ Data loaded and cleaned!')


## Calculate Correlation Matrix

In [ ]:
# Calculate correlations
corr_matrix = calculate_correlation_matrix(clean_stocks)

print('\nCorrelation Matrix:')
print(corr_matrix.round(3))


## Visualize Correlation Heatmap

In [ ]:
# Plot heatmap
plot_correlation_heatmap(corr_matrix)


## Identify High Correlations

In [ ]:
# Find high correlations (threshold > 0.7)
high_corr = identify_correlations(corr_matrix, threshold=0.7)

print('\n' + '='*80)
print('HIGHLY CORRELATED STOCK PAIRS (> 0.7)')
print('='*80)

if high_corr:
    for pair in high_corr:
        print(f'{pair["Stock1"]} <-> {pair["Stock2"]}: {pair["Correlation"]:.3f}')
else:
    print('No pairs with correlation > 0.7')

# Find moderate correlations (threshold > 0.5)
mod_corr = identify_correlations(corr_matrix, threshold=0.5)

print('\n' + '='*80)
print('MODERATELY CORRELATED PAIRS (> 0.5)')
print('='*80)

if mod_corr:
    for pair in mod_corr:
        print(f'{pair["Stock1"]} <-> {pair["Stock2"]}: {pair["Correlation"]:.3f}')
else:
    print('No pairs with correlation > 0.5')


## Analyze MSFT Correlations

In [ ]:
print('\n' + '='*80)
print(f'MSFT CORRELATION WITH OTHER STOCKS')
print('='*80)

msft_corr = corr_matrix[PRIMARY_STOCK].drop(PRIMARY_STOCK).sort_values(ascending=False)

for stock, corr_value in msft_corr.items():
    print(f'{stock}: {corr_value:.3f}')


## Price Movement Comparison

In [ ]:
# Create moving average comparison
fig, ax = plt.subplots(figsize=(14, 6))

for ticker in ALL_STOCKS:
    # Normalize to 100 at start
    normalized = (clean_stocks[ticker]['Close'] / clean_stocks[ticker]['Close'].iloc[0]) * 100
    ax.plot(normalized.index, normalized, label=ticker, linewidth=2)

ax.set_title('Normalized Price Performance Comparison (1 Year)', fontsize=14, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Normalized Price (Base = 100)', fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../visualizations/correlation_performance_comparison.png', dpi=300, bbox_inches='tight')
print('Saved: visualizations/correlation_performance_comparison.png')
plt.show()


## Key Insights from Correlation Analysis

In [ ]:
print('\n' + '='*80)
print('CORRELATION INSIGHTS')
print('='*80)

print(f'\n1. MSFT is most correlated with: {msft_corr.idxmax()} ({msft_corr.max():.3f})')
print(f'2. MSFT is least correlated with: {msft_corr.idxmin()} ({msft_corr.min():.3f})')

print('\n3. Interpretation:')
print('   - Strong correlation (>0.7) indicates stocks move together')
print('   - Moderate correlation (0.5-0.7) indicates related movements')
print('   - Tech stocks often correlate due to similar market factors')

print('\n✅ Correlation analysis complete! Proceed to predictive modeling.')
